# 02 — The Python API

So far we've driven the package through its CLI (`python -m wcl_data …`).
Under the hood the CLI is just a thin argparse wrapper around a handful
of public Python functions and classes — and you can call those directly
from your own scripts, notebooks, or orchestration code.

This chapter is a tour of the public surface:

- `load_settings()` and the `Settings` dataclass,
- `open_db()` and the `Repository` class with its transaction context manager,
- `APIClient` and its streaming retry semantics,
- the fetcher orchestrators (`refresh_all`, `pull_new`, `hydrate_entity`),
- the event-location parser.

> Cells in this notebook do hit the API in section 8, but only for a tiny
> sample (5 athletes), so the whole notebook runs in well under a minute.


## 1. Working directory

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("cwd:", Path.cwd())


## 2. Load settings from `.env`

Every entry point of the package starts with `load_settings()` —
it reads `.env` (via python-dotenv), validates the values, and
returns a frozen `Settings` dataclass.

Use `require_credentials=False` if you're going to do read-only work
on the DB and don't want to bother with `WCL_CSRF_TOKEN` /
`WCL_SESSION_COOKIE` being set.

In [ ]:
from wcl_data.config import load_settings

settings = load_settings()  # require_credentials=True by default
print(f"db_path        = {settings.db_path}")
print(f"max_workers    = {settings.max_workers}")
print(f"connect_timeout= {settings.connect_timeout}s")
print(f"read_timeout   = {settings.read_timeout}s")
print(f"stale_days     = {settings.stale_days}")
print(f"referer        = {settings.referer}")
print(f"csrf_token     = {'(set, %d chars)' % len(settings.csrf_token)}")
print(f"session_cookie = {'(set, %d chars)' % len(settings.session_cookie)}")


### The `api_headers` property

`Settings.api_headers` returns the three headers the API client sends on
every request. The package never inlines these — it always goes through
this property so you can override them in tests.

In [ ]:
# Don't print the actual values (they're secrets), just the keys.
print(sorted(settings.api_headers.keys()))


## 3. Opening a `Repository`

`Repository` wraps a `sqlite3.Connection` with typed methods, one per
entity. It also exposes the staleness logic (`find_stale`, `stale_cutoff`)
and the transaction context manager.

In [ ]:
from wcl_data.db.schema import open_db
from wcl_data.db.repository import Repository

conn = open_db(settings.db_path)
repo = Repository(conn)

# `count` and `count_hydrated` are the generic helpers.
print(f"{'table':16s}  {'rows':>8s}  {'hydrated':>10s}")
for tbl in ("seasons", "events", "competitions", "athletes", "results"):
    rows = repo.count(tbl)
    if tbl == "results":
        print(f"{tbl:16s}  {rows:>8d}  {'—':>10s}")
    else:
        print(f"{tbl:16s}  {rows:>8d}  {repo.count_hydrated(tbl):>10d}")


## 4. The transaction context manager

Every `upsert_*` and `update_*` method commits as it goes — that's what
makes streaming fetches durable.

For batches that must be atomic (insert several rows or none), wrap them
in `repo.transaction()`. Inside the `with`, per-call commits are
suppressed; the context manager commits once at the end, or rolls back
if any exception propagates.

In [ ]:
# Demonstrate rollback semantics on a throwaway in-memory DB so we don't
# touch the real warehouse.
import sqlite3
from wcl_data.db.schema import apply_schema

scratch = sqlite3.connect(":memory:")
scratch.row_factory = sqlite3.Row
apply_schema(scratch)
scratch_repo = Repository(scratch)

print("before:", scratch_repo.count("leagues"))

try:
    with scratch_repo.transaction():
        scratch_repo.upsert_league("World Cup")
        scratch_repo.upsert_league("Continental Championships")
        raise RuntimeError("oops — something went wrong mid-batch")
except RuntimeError as exc:
    print("caught:", exc)

print("after :", scratch_repo.count("leagues"), "(should still be 0 — rolled back)")
scratch.close()


## 5. Finding stale rows

`find_stale("athletes", stale_days=30)` returns every athlete row whose
`last_fetched_at` is either NULL or older than the cutoff. NULL counts
as stale, so skeleton rows show up automatically.

The boundary is **exclusive of equality at the second** — a row stamped
exactly `stale_days` ago is still considered fresh.

In [ ]:
# What fraction of athletes is currently stale at the 30-day threshold?
stale = repo.find_stale("athletes", stale_days=30)
total = repo.count("athletes")
print(f"{len(stale):>6d} / {total} athletes considered stale at stale_days=30")
print(f"{'':>6s}   ({100 * len(stale) / total:.1f}% of the table)")


You can use `find_stale` directly to write your own refresh loops —
that's exactly what the fetcher modules in `src/wcl_data/fetchers/`
do.

## 6. The streaming `APIClient`

`APIClient` is a thin layer over `requests.Session` with three notable
features:

1. A connection pool sized to `max_workers`, so urllib3 doesn't warn
   under heavy concurrency.
2. A `ThreadPoolExecutor` that fetches a list of IDs concurrently and
   **yields** results as they finish — you can write each one to disk
   as it arrives instead of waiting for the whole batch.
3. Selective retry — 5xx, 429 (with `Retry-After` honored), and
   transport errors are retried with exponential backoff + jitter;
   other 4xx is treated as permanent (so a 404 on a missing athlete
   doesn't burn retry budget). Five consecutive 401/403 across the
   pool trips `AuthFailureAbort` — the run halts loudly instead of
   silently dropping every remaining row.

In [ ]:
from wcl_data.api.client import APIClient, Fetched, FetchError

client = APIClient(settings)
print(client)
print()
print("settings.api_headers keys:", sorted(client._session.headers.keys() & {'X-Csrf-Token', 'Cookie', 'Referer'}))


### Live fetch: 5 athletes, streamed

`client.stream(endpoint, ids)` yields `Fetched(key, path, data)` items as
each request completes. `key` is the original ID, `path` is the URL path
hit, `data` is the parsed JSON dict.

In [ ]:
# Pick five real athlete ifsc_ids from the warehouse.
sample_ids = [r['ifsc_id'] for r in conn.execute(
    "SELECT ifsc_id FROM athletes ORDER BY id LIMIT 5"
)]
print("fetching:", sample_ids)
print()

for fetched in client.stream("athletes", sample_ids, max_retries=1):
    name = f"{fetched.data.get('firstname', '?')} {fetched.data.get('lastname', '?')}"
    print(f"  id={fetched.key:<6d}  path={fetched.path:<24s}  → {name}")


## 7. Retry semantics in action

The default policy retries 5xx / transport errors and gives up on 4xx
after a single attempt. You can override the predicate with the
`retry_on` kwarg — useful in tests or when you want to retry 4xx for
a specific reason.

In [ ]:
# Inspect the default predicate
from wcl_data.api.client import _default_retry_on

mock_5xx = FetchError("HTTP 503 Service Unavailable", status_code=503)
mock_429 = FetchError("HTTP 429 Too Many Requests", status_code=429)
mock_404 = FetchError("HTTP 404 Not Found", status_code=404)
mock_net = FetchError("ConnectionError", status_code=None)

for label, err in [("503 (5xx)", mock_5xx), ("429 rate-limit", mock_429),
                   ("404 (4xx)", mock_404), ("transport", mock_net)]:
    print(f"  {label:<16s}  retry? {_default_retry_on(err)}")


## 8. Calling a fetcher orchestrator directly

`src/wcl_data/fetchers/refresh.py` exposes three orchestrators that
combine discovery + hydration:

- `refresh_all(repo, client, *, stale_days, limit=None)` — drive the
  full pipeline, per-entity stale check. The everyday `refresh` command.
- `pull_new(repo, client, *, limit=None)` — force-refresh containers,
  hydrate only newly-discovered athletes. The everyday `pull-new`
  command.
- `hydrate_entity(repo, client, entity, *, stale_days, limit=None)` —
  hydrate a single entity. The `hydrate <entity>` command.

Each returns a counts dict (or tuple) you can log or assert on.

Let's run `hydrate_entity` on athletes with `limit=5` — small enough
to finish quickly, big enough to demonstrate.

In [ ]:
from wcl_data.fetchers.refresh import hydrate_entity

ok, fail = hydrate_entity(repo, client, "athletes", stale_days=0, limit=5)
print(f"ok={ok}  fail={fail}")


`stale_days=0` means "treat everything as stale" — so this re-fetches
5 athletes regardless of their `last_fetched_at`. In a real script
you'd use `stale_days=30` (or whatever's in your `.env`).

## 9. The event location parser

Event names from the World Climbing API embed city and country info, but the
format is inconsistent. The parser at
`src/wcl_data/parsers/event_location.py` applies a series of
heuristics — country code in parentheses, country name, country code
before a year, etc. — and returns `(city, ISO3 country)` or `(None, None)`
if nothing extractable.

The parser is **conservative by design**: it returns `(None, None)`
rather than guess. About 4% of events end up with NULL city or country
for that reason.

In [ ]:
from wcl_data.parsers.event_location import parse_city_country

samples = [
    "IFSC Climbing World Cup (Innsbruck, AUT) 2024",
    "IFSC Climbing World Championships (Bern) 2023",
    "IFSC World Cup Salt Lake City (USA) 2022",
    "Some Random Event Name Without Location Info",
]

for name in samples:
    city, country = parse_city_country(name)
    print(f"  {name!r:<55s}  → city={city!r}, country={country!r}")


### The cross-batch country backfill

For events where the parser couldn't extract a country but the city is
known and appears elsewhere with a country, `Repository.backfill_event_country_from_siblings()`
fills in the NULLs from sibling rows. This runs after the events
hydration step in `refresh_all` / `pull_new`.

It's idempotent — you can call it at any time and it's cheap (one
SQL `UPDATE … WHERE EXISTS`).

In [ ]:
# Just demonstrate the call. The warehouse has already been backfilled,
# so the affected count is usually 0 here.
affected = repo.backfill_event_country_from_siblings()
print(f"backfilled {affected} event rows")


## 10. Clean up

In [ ]:
conn.close()


## What's next

Chapter 03 — [`03_querying_and_exporting.ipynb`](03_querying_and_exporting.ipynb) —
shows three real analytical queries on the warehouse and walks through
the seven pre-joined export views (plus the opt-in `ascents` view) —
the data you'd hand to a downstream consumer.
